<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/5%20-%20Exemplos%20de%20Aplica%C3%A7%C3%A3o/5.1_doacao_de_sangue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 5.1 — Doação de sangue: classificação com rede densa

Os notebooks do módulo 5 aplicam o pipeline completo do módulo 4 — `Dataset` e `DataLoader`, rede com camadas escondidas, dropout, Adam e avaliação em dados que a rede não viu no treino — a problemas diferentes do MNIST e da Iris.

Neste primeiro exemplo, o problema é prever se um doador de sangue voltou a doar em março de 2007, a partir do seu histórico: há quantos meses foi a última doação, quantas doações fez e há quantos meses foi a primeira. A entrada é um vetor de 3 características (features) de cada doador; a saída são 2 logits, um por classe (não doou / doou).

## Objetivos

- Preparar um dataset tabular real: inspecionar as colunas, remover uma feature redundante e padronizar as features sem vazamento de informação do teste.
- Treinar uma rede densa com dropout, Adam, `StepLR` (redução da taxa de aprendizado) e parada antecipada (early stopping), salvando os pesos da melhor época.
- Avaliar um classificador em dados desbalanceados: comparar a acurácia com o baseline da classe majoritária e olhar a matriz de confusão e o recall da classe positiva.

## Importação das bibliotecas

In [ ]:
import os
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

torch.manual_seed(1111)   # reprodutibilidade: inicialização dos pesos, dropout e embaralhamento
np.random.seed(1111)

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print("device:", device)

## Dados

Usamos o dataset [Blood Transfusion Service Center](https://archive.ics.uci.edu/dataset/176/blood+transfusion+service+center), do repositório UCI. São dados reais de 748 doadores do Centro de Transfusão de Sangue da cidade de Hsin-Chu, em Taiwan, publicados por Yeh, Yang e Ting em [*Knowledge discovery on RFM model using Bernoulli sequence*](https://doi.org/10.1016/j.eswa.2008.07.018) (Expert Systems with Applications, 36(3), 2009). O arquivo baixa sem login; se o endereço principal falhar, a célula tenta o zip da página do dataset.

As colunas seguem o modelo RFM (recência, frequência, valor monetário), usado em marketing para prever se um cliente vai voltar, acrescido do tempo desde a primeira doação:

| coluna original | nome aqui | significado |
|---|---|---|
| `Recency (months)` | `recencia` | meses desde a última doação |
| `Frequency (times)` | `frequencia` | número total de doações |
| `Monetary (c.c. blood)` | `volume` | volume total doado, em cm³ |
| `Time (months)` | `tempo` | meses desde a primeira doação |
| `whether he/she donated blood in March 2007` | `doou` | alvo: 1 se doou em março de 2007, 0 se não |

In [ ]:
# Baixa o CSV do repositório UCI (não exige login) para ./data.
os.makedirs("data", exist_ok=True)
csv_path = "data/transfusion.data"
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/blood-transfusion/transfusion.data"
url_zip = "https://archive.ics.uci.edu/static/public/176/blood+transfusion+service+center.zip"   # alternativa
if not os.path.exists(csv_path):
    try:
        urllib.request.urlretrieve(url, csv_path)
    except Exception as erro:
        print("endereço principal falhou:", erro, "| tentando o zip")
        urllib.request.urlretrieve(url_zip, "data/transfusion.zip")
        with zipfile.ZipFile("data/transfusion.zip") as z:
            z.extract("transfusion.data", "data")

df = pd.read_csv(csv_path)
df.columns = ["recencia", "frequencia", "volume", "tempo", "doou"]   # nomes curtos em português

In [ ]:
df.head()

In [ ]:
df.describe()

O alvo é **desbalanceado**: a maioria dos doadores não doou em março de 2007. Guarde a proporção da classe majoritária: um classificador que sempre responde "não doou" já acerta essa fração.

In [ ]:
print(df["doou"].value_counts())
print(df["doou"].value_counts(normalize=True).round(4))

### Preparação das features

Todas as colunas já são numéricas, mas uma delas é redundante. Cada doação corresponde a 250 cm³ de sangue, então `volume` é sempre 250 × `frequencia`: as duas colunas têm correlação 1 (colinearidade perfeita) e `volume` não acrescenta informação. Removemos `volume` e ficamos com 3 features.

In [ ]:
print("volume == 250 * frequencia em todas as linhas:", (df["volume"] == 250 * df["frequencia"]).all())
print("correlação entre volume e frequencia:", df["volume"].corr(df["frequencia"]).round(4))

df = df.drop(columns=["volume"])
print(df.shape)

Antes de treinar, vale olhar a correlação de cada feature com o alvo. A correlação mede só relações lineares, mas dá uma primeira ideia de quais features carregam informação.

In [ ]:
correlation = df.corr()["doou"].drop("doou").sort_values()

plt.figure(figsize=(7, 3))
plt.barh(correlation.index, correlation.values)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlação de cada feature com 'doou'")
plt.xlabel("correlação de Pearson")
plt.tight_layout()
plt.show()

print(correlation.round(4))

`recencia` tem a correlação mais forte com o alvo (−0,28): quem doou há pouco tempo tende a voltar. `frequencia` vem em seguida (+0,22): quem já doou muitas vezes tende a continuar. `tempo` sozinho quase não se correlaciona com o alvo (−0,04), mas pode ajudar combinado com as outras (um doador antigo com poucas doações é diferente de um doador novo com poucas doações). Há sinal nos dados, mas moderado.

Separamos as features (`X`) do alvo (`Y`):

In [ ]:
target = "doou"
features = [c for c in df.columns if c != target]

X = df[features].values.astype(np.float64)
Y = df[target].values
print("features:", features)
print("X:", X.shape, "| Y:", Y.shape)

### Treino, teste e padronização

Separamos 70% dos doadores para treino e 30% para teste. Com `stratify=Y`, a divisão é **estratificada**: treino e teste ficam com a mesma proporção de "doou" do dataset completo. Sem isso, com poucas amostras da classe positiva, um dos conjuntos poderia ficar com proporção bem diferente só por acaso. `random_state=42` fixa o sorteio.

As features têm escalas muito diferentes (`frequencia` vai de 1 a 50, `tempo` de 2 a 98). O `StandardScaler` subtrai a média e divide pelo desvio-padrão de cada feature; ele é ajustado (`fit`) **só no treino** e depois aplicado ao teste, para que nenhuma informação do teste (média, desvio) entre no treino.

Neste notebook, o conjunto de teste também é usado para escolher a melhor época (parada antecipada). Na prática, ele faz o papel de conjunto de validação; uma avaliação final sem viés exigiria um terceiro conjunto, separado desde o início.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42, stratify=Y)

X_scaler = StandardScaler()
X_train = X_scaler.fit_transform(X_train)   # média e desvio calculados só no treino
X_test = X_scaler.transform(X_test)         # o teste usa as estatísticas do treino
print("treino:", X_train.shape, "| teste:", X_test.shape)
print(f"proporção de 'doou' | treino: {y_train.mean():.3f} | teste: {y_test.mean():.3f}")

O `Dataset` é o mesmo padrão do notebook 3.3: `__len__` devolve o número de amostras e `__getitem__` devolve o par (features, rótulo). As features viram `float32` e os rótulos, inteiros `int64` (`torch.long`), o tipo que `nn.CrossEntropyLoss` espera para a classe correta.

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [ ]:
train_dataset = TabularDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_dataset = TabularDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))
train_dataset[0]

In [ ]:
train_kwargs = {"batch_size": 100, "shuffle": True}    # embaralha o treino a cada época
test_kwargs = {"batch_size": 100, "shuffle": False}    # a ordem não importa na avaliação
if use_cuda:
    cuda_kwargs = {"num_workers": 1, "pin_memory": True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

train_loader = DataLoader(train_dataset, **train_kwargs)
test_loader = DataLoader(test_dataset, **test_kwargs)
print("mini-batches por época:", len(train_loader))

## Modelo

Rede densa com 3 entradas, duas camadas escondidas (128 e 256 neurônios, ReLU) e uma camada de saída com 2 logits. É uma rede grande para 3 features e 523 doadores de treino; o dropout ajuda a conter o overfitting.

Entre as camadas há **dropout** (`nn.Dropout(0.3)`): durante o treino, cada ativação é zerada com probabilidade 0,3, sorteada a cada mini-batch. Isso impede a rede de depender demais de poucos neurônios e funciona como regularização. Na avaliação, o dropout é desligado. É por isso que `model.train()` e `model.eval()` aparecem nas funções de treino e teste: eles trocam o modo das camadas que se comportam de forma diferente nas duas fases.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.input = nn.Linear(X_train.shape[1], 128)
        self.drop1 = nn.Dropout(0.3)
        self.hidden = nn.Linear(128, 256)
        self.drop2 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, 2)

    def forward(self, x):
        x = self.input(x)
        x = F.relu(x)
        x = self.drop1(x)
        x = self.hidden(x)
        x = F.relu(x)
        x = self.drop2(x)
        x = self.fc2(x)      # logits: sem softmax (a CrossEntropyLoss já aplica log-softmax)
        return x

model = Net()
print(model)
print("parâmetros:", sum(p.numel() for p in model.parameters()))

## Treinamento

`train` percorre uma época de mini-batches atualizando os pesos e devolve o custo médio e a acurácia no treino. `test` só avalia, sem gradientes, e devolve as mesmas medidas no teste.

In [ ]:
def train(model, device, train_loader, optimizer, criterion):
    model.train()                       # liga o dropout
    running_loss = 0.0
    correct = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)            # logits, shape (N, n_classes)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data.size(0)    # loss é a média do mini-batch; multiplicamos pelo tamanho para somar
        pred = output.argmax(dim=1)                   # classe com o maior logit
        correct += (pred == target).sum().item()
    train_loss = running_loss / len(train_loader.dataset)
    train_acc = 100 * correct / len(train_loader.dataset)
    return train_loss, train_acc

In [ ]:
def test(model, device, test_loader, criterion):
    model.eval()                        # desliga o dropout
    test_loss = 0.0
    correct = 0
    with torch.no_grad():               # avaliação: sem gradientes
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() * data.size(0)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
    test_loss /= len(test_loader.dataset)
    test_acc = 100 * correct / len(test_loader.dataset)
    return test_loss, test_acc

O laço de treino acrescenta três peças ao que vimos no módulo 4:

- `StepLR(optimizer, step_size=lr_decay, gamma=0.5)`: um agendador (scheduler) que, a cada `lr_decay` épocas, multiplica a taxa de aprendizado (learning rate) por 0,5. `scheduler.step()` é chamado uma vez por época.
- **Parada antecipada (early stopping)**: se a acurácia de teste não melhora por `patience` épocas seguidas, o treino para.
- **Checkpoint**: sempre que a acurácia de teste melhora, `torch.save(model.state_dict(), ckpt_path)` grava os pesos. O `state_dict` é um dicionário com os tensores de pesos e bias de cada camada.

In [ ]:
n_epochs = 100
lr = 0.001
lr_decay = 10     # a cada lr_decay épocas, o StepLR multiplica a taxa de aprendizado por gamma=0.5
patience = 10      # parada antecipada: para se a acurácia de teste não melhorar por `patience` épocas
patience_counter = 0
best_acc = 0
ckpt_path = "melhor_modelo_doacao.pt"

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

torch.manual_seed(1111)
model = Net().to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()   # recebe logits (aplica log-softmax internamente)
scheduler = StepLR(optimizer, step_size=lr_decay, gamma=0.5)

print("Iniciando treinamento ...")
for epoch in range(1, n_epochs + 1):
    train_loss, train_acc = train(model, device, train_loader, optimizer, criterion)
    test_loss, test_acc = test(model, device, test_loader, criterion)
    scheduler.step()

    print(f"Época {epoch:3d} | treino: custo {train_loss:.4f}, acc {train_acc:.2f}% | teste: custo {test_loss:.4f}, acc {test_acc:.2f}%")
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), ckpt_path)   # state_dict: dicionário com os tensores de pesos e bias
        print("    melhor época até agora: pesos salvos")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter == patience:
            print(f"Parada antecipada na época {epoch}")
            break

In [ ]:
# Recarrega os pesos da melhor época (os da última época podem ser piores)
model.load_state_dict(torch.load(ckpt_path, map_location=device))
test_loss, test_acc = test(model, device, test_loader, criterion)
print(f"Melhor modelo | teste: custo {test_loss:.4f}, acurácia {test_acc:.2f}%")

## Avaliação

In [ ]:
plt.figure(figsize=(12, 4.5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], label="treino")
plt.plot(range(1, len(history["test_loss"]) + 1), history["test_loss"], label="teste")
plt.title("Função de custo (cross-entropy)")
plt.xlabel("época")
plt.ylabel("custo")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(history["train_acc"]) + 1), history["train_acc"], label="treino")
plt.plot(range(1, len(history["test_acc"]) + 1), history["test_acc"], label="teste")
plt.title("Acurácia")
plt.xlabel("época")
plt.ylabel("acurácia (%)")
plt.legend()
plt.grid(True)

plt.show()

### Baseline e matriz de confusão

Como cerca de 76% dos doadores não doaram, a acurácia precisa ser comparada com o **baseline da classe majoritária**: um "modelo" que ignora as features e sempre prevê a classe mais frequente no treino.

In [ ]:
classe_majoritaria = int(y_train.mean() >= 0.5)            # 0: "não doou" é a maioria no treino
baseline_acc = 100 * (y_test == classe_majoritaria).mean()
print(f"baseline (sempre prevê {classe_majoritaria}): {baseline_acc:.2f}%")
print(f"rede (melhor época):         {test_acc:.2f}%")

A acurácia sozinha esconde **como** a rede erra. A matriz de confusão conta, para cada classe real (linhas), quantas amostras foram previstas em cada classe (colunas). O **recall** (sensibilidade) da classe "doou" é a fração dos doadores que doaram e que a rede identificou:

$$\text{recall} = \frac{VP}{VP + FN}$$

onde VP (verdadeiros positivos) são os que doaram e foram previstos como "doou" e FN (falsos negativos) são os que doaram e foram previstos como "não doou". O baseline tem recall 0: nunca acerta um doador.

In [ ]:
def predict(model, loader):
    # Devolve as classes previstas para todas as amostras do loader, na ordem do loader
    model.eval()
    preds = []
    with torch.no_grad():
        for data, _ in loader:
            preds.append(model(data.to(device)).argmax(dim=1).cpu())
    return torch.cat(preds).numpy()

y_pred = predict(model, test_loader)        # test_loader não embaralha: mesma ordem de y_test
cm = confusion_matrix(y_test, y_pred)       # linhas: classe real; colunas: classe prevista
ConfusionMatrixDisplay(cm, display_labels=["não doou", "doou"]).plot(cmap="Blues")
plt.title("Matriz de confusão (teste)")
plt.xlabel("previsto")
plt.ylabel("real")
plt.show()

recall_doou = cm[1, 1] / cm[1].sum()
print(f"recall da classe 'doou': {recall_doou:.2%} ({cm[1, 1]} de {cm[1].sum()})")

A rede acerta 80,00% no teste, contra 76,00% do baseline: um ganho real, mas pequeno (9 doadores a mais, em 225). A matriz de confusão mostra o porquê: a rede identifica só 19 dos 54 doadores que doaram (recall de 35,19%) e erra 10 dos 171 que não doaram. Como "não doou" é a classe majoritária, a rede só prevê "doou" quando os sinais são fortes (doação recente e frequente). Nas curvas, os custos de treino e de teste ficam próximos (≈ 0,46) e estáveis a partir da época 6, sem overfitting evidente; a melhor época foi a 6 e a parada antecipada interrompeu o treino na época 16.

## Exercícios

**1.** Na matriz de confusão `cm`, calcule a **precisão** da classe "doou": entre os doadores que a rede previu como "doou", que fração doou de fato? Fórmula: $\text{precisão} = \frac{VP}{VP + FP}$, onde FP (falsos positivos) são os que não doaram mas foram previstos como "doou". Para um banco de sangue que vai telefonar para os doadores previstos como "doou", o que a precisão e o recall significam?

In [ ]:
# TODO: escreva sua solução aqui
print(f"precisão da classe 'doou': {precisao_doou:.2%}")
print(f"recall da classe 'doou':   {recall_doou:.2%}")

*Escreva sua resposta aqui.*

**2.** A `nn.CrossEntropyLoss` aceita um argumento `weight`, com um peso por classe: o custo de cada amostra é multiplicado pelo peso da sua classe. Dê à classe "doou" um peso maior, inversamente proporcional à sua frequência no treino:

$$w_c = \frac{N}{2 N_c}$$

onde $N$ é o total de amostras de treino e $N_c$ o número de amostras da classe $c$. Treine um novo modelo `Net()` por 30 épocas com esse custo (use a função `train`, sem parada antecipada) e compare acurácia e recall de "doou" com os do modelo anterior.

In [ ]:
# TODO: escreva sua solução aqui
print(cm_w)
print(f"custo ponderado | acurácia {acc_w:.2f}% | recall 'doou' {recall_w:.2%}")
print(f"custo original  | acurácia {test_acc:.2f}% | recall 'doou' {recall_doou:.2%}")

*Escreva sua resposta aqui.*

**3.** A rede tem 34.050 parâmetros para 3 features. Com base no baseline, na matriz de confusão e nas curvas de treino e teste, quanto a rede aprendeu? O que você testaria antes de concluir que uma rede neural é a melhor escolha para este problema?

*Escreva sua resposta aqui.*

## Aprendizados

- Antes de modelar, inspecione as colunas: `volume` = 250 × `frequencia` é uma feature redundante (colinear) e foi removida.
- O split estratificado mantém a proporção das classes em treino e teste; o `StandardScaler` é ajustado só no treino e aplicado ao teste.
- Dropout, `StepLR`, parada antecipada e checkpoint (`state_dict`) completam o pipeline de treino; usar o teste para escolher a época faz dele um conjunto de validação.
- Em dados desbalanceados, compare a acurácia com o baseline da classe majoritária; a matriz de confusão, o recall e a precisão mostram como o modelo erra, e o argumento `weight` da `CrossEntropyLoss` troca acurácia por recall.